# Capilar — Macenko, mesma receita do v4

Normaliza os tiles de capilar (train e valid) com `torchstain.MacenkoNormalizer` para uma lâmina ROSILHA e treina YOLO11s-seg de novo. Não mexe no microcotilédone nem no `best.pt` do v4. Dataset fica em `D:/projeto_placentas_clayton/dataset_v3.1_yolo11_tiled_3x3_macenko`.

Critério: F1 tile @0.33 contra **0.726** do v4. Abaixo de +2 pontos, manter o v4.

In [1]:
from __future__ import annotations

import json
import os
import shutil
import time
from pathlib import Path

_TMP_D = Path(r'D:\projeto_placentas_clayton\temp_ml')
_TMP_D.mkdir(parents=True, exist_ok=True)
os.environ['TEMP'] = str(_TMP_D)
os.environ['TMP'] = str(_TMP_D)
os.environ['TMPDIR'] = str(_TMP_D)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

_ULTRA_SETTINGS = Path(os.environ.get('APPDATA', '')) / 'Ultralytics' / 'settings.json'
if _ULTRA_SETTINGS.is_file():
    _cfg = json.loads(_ULTRA_SETTINGS.read_text(encoding='utf-8'))
    _cfg['datasets_dir'] = r'D:\projeto_placentas_clayton\datasets_ultralytics'
    _cfg['weights_dir'] = r'D:\projeto_placentas_clayton\weights_ultralytics'
    _cfg['runs_dir'] = r'D:\projeto_placentas_clayton\runs_ultralytics'
    _ULTRA_SETTINGS.write_text(json.dumps(_cfg, indent=2), encoding='utf-8')

import cv2
import numpy as np
import torch
import ultralytics
import yaml
from importlib.metadata import version
from torchstain.base.normalizers import MacenkoNormalizer
from ultralytics import YOLO

print(f'TEMP/TMP -> {_TMP_D}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'torchstain: {version("torchstain")}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

TEMP/TMP -> D:\projeto_placentas_clayton\temp_ml
Ultralytics: 8.4.155
PyTorch: 2.5.1+cu121
torchstain: 1.4.1
GPU: NVIDIA GeForce GTX 1650 SUPER


In [2]:
def discover_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists() and (p / 'v2').exists():
            return p
        p = p.parent
    raise RuntimeError('Repo root not found (.git + v2)')

REPO_ROOT = discover_repo_root()

CFG = {
    'src_dataset': Path(r'D:/projeto_placentas_clayton/dataset_v3.1_yolo11_tiled_3x3'),
    'dst_dataset': Path(r'D:/projeto_placentas_clayton/dataset_v3.1_yolo11_tiled_3x3_macenko'),
    'data_yaml': 'v3_capilar_yolo11s/data_capilar_tiled_macenko.yaml',
    'pretrained': 'yolo11s-seg.pt',
    'model_name': 'yolo11s-seg',
    'project': 'v3_capilar_yolo11s/runs',
    'run_name': 'capilar_yolo11s_tiled_macenko_v1',
    'epochs': 80,
    'imgsz': 640,
    'batch': 1,
    'workers': 0,
    'patience': 30,
    'max_det': 150,
    'classes': [0],
    'iou_match': 0.5,
    'area_factor': (50 / 72) ** 2 * (640 / 4140) ** 2,
    'conf': 0.33,
    'output_root': 'v3_capilar_yolo11s/artifacts_macenko',
    'v4_f1': 0.7258003048780487,
    'v4_iou': 0.7950624685220785,
    'v4_map50': 0.778,
}

data_yaml = (REPO_ROOT / CFG['data_yaml']).resolve()
out_root = (REPO_ROOT / CFG['output_root']).resolve()
out_bench = out_root / 'benchmarks'
out_preview = out_root / 'stain_preview'
for p in (out_root, out_bench, out_preview):
    p.mkdir(parents=True, exist_ok=True)

pretrained = Path(CFG['pretrained'])
if not pretrained.is_file():
    pretrained = (REPO_ROOT / CFG['pretrained']).resolve()
if not pretrained.is_file():
    wdir = Path(r'D:\projeto_placentas_clayton\weights_ultralytics') / CFG['pretrained']
    if wdir.is_file():
        pretrained = wdir

print('repo:', REPO_ROOT)
print('src:', CFG['src_dataset'])
print('dst:', CFG['dst_dataset'])
print('pretrained:', pretrained, 'exists=', pretrained.is_file())
print('run_name:', CFG['run_name'])

repo: D:\projeto_placentas_clayton\dev\projeto-placentas
src: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3
dst: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3_macenko
pretrained: D:\projeto_placentas_clayton\dev\projeto-placentas\yolo11s-seg.pt exists= True
run_name: capilar_yolo11s_tiled_macenko_v1


In [3]:
def _tissue_pixels(img_rgb: np.ndarray) -> int:
    flat = img_rgb.reshape(-1, 3).astype(np.float64)
    od = -np.log((flat + 1) / 240.0)
    return int((~np.any(od < 0.15, axis=1)).sum())


def apply_macenko(img_rgb: np.ndarray, normalizer) -> np.ndarray | None:
    try:
        out, _, _ = normalizer.normalize(img_rgb, stains=False)
    except Exception:
        return None
    if out is None or out.shape != img_rgb.shape:
        return None
    return out


def _read_rgb(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f'falha ao ler {path}')
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def _write_jpg(path: Path, img_rgb: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ok = cv2.imwrite(str(path), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
    if not ok:
        raise RuntimeError(f'falha ao gravar {path}')


src_root = CFG['src_dataset']
dst_root = CFG['dst_dataset']
ref_json = dst_root / 'macenko_reference.json'
assert src_root.is_dir(), src_root

cands = sorted((src_root / 'train' / 'images').glob('ROSILHA*r1c1.jpg'))
if not cands:
    raise RuntimeError('nenhum tile ROSILHA*r1c1.jpg no treino')
best_path, best_n, best_img = None, -1, None
for cand in cands:
    img = _read_rgb(cand)
    n_tissue = _tissue_pixels(img)
    if n_tissue > best_n:
        best_path, best_n, best_img = cand, n_tissue, img

normalizer = MacenkoNormalizer(backend='numpy')
normalizer.fit(best_img)
dst_root.mkdir(parents=True, exist_ok=True)
ref_meta = {
    'backend': 'torchstain',
    'source': str(best_path),
    'tissue_pixels': best_n,
    'HE': np.asarray(normalizer.HERef).tolist(),
    'maxC': np.asarray(normalizer.maxCRef).tolist(),
}
ref_json.write_text(json.dumps(ref_meta, indent=2), encoding='utf-8')
print('referencia:', best_path.name, 'tissue_px', best_n)
print('HE', np.round(np.asarray(normalizer.HERef), 3))
print('maxC', np.round(np.asarray(normalizer.maxCRef), 3))

preview_globs = ('ROSILHA*r1c1.jpg', 'TOSTADA*r1c1.jpg', 'ZAZA*r1c1.jpg')
for pattern in preview_globs:
    hits = sorted((src_root / 'train' / 'images').glob(pattern))
    if not hits:
        print('preview ausente:', pattern)
        continue
    src_img = _read_rgb(hits[0])
    norm = apply_macenko(src_img, normalizer)
    shown = src_img if norm is None else norm
    pair = np.hstack([src_img, shown])
    scale = 1400 / pair.shape[1]
    pair = cv2.resize(pair, (1400, int(pair.shape[0] * scale)), interpolation=cv2.INTER_AREA)
    out_name = out_preview / f'{hits[0].stem[:40]}_before_after.jpg'
    _write_jpg(out_name, pair)
    print('preview:', out_name.name, 'normalized' if norm is not None else 'copied')

n_norm = n_copy = n_skip = 0
t0 = time.perf_counter()
jobs = []
for split in ('train', 'valid'):
    for img_path in sorted((src_root / split / 'images').glob('*.jpg')):
        jobs.append((split, img_path))

for i, (split, img_path) in enumerate(jobs, start=1):
    dst_img = dst_root / split / 'images' / img_path.name
    dst_lbl = dst_root / split / 'labels' / f'{img_path.stem}.txt'
    src_lbl = src_root / split / 'labels' / f'{img_path.stem}.txt'
    if dst_img.is_file() and dst_img.stat().st_size > 0 and dst_lbl.is_file():
        n_skip += 1
        continue
    dst_img.parent.mkdir(parents=True, exist_ok=True)
    dst_lbl.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_lbl, dst_lbl)
    rgb = _read_rgb(img_path)
    norm = apply_macenko(rgb, normalizer)
    if norm is None:
        shutil.copy2(img_path, dst_img)
        n_copy += 1
    else:
        _write_jpg(dst_img, norm)
        n_norm += 1
    if i % 100 == 0 or i == len(jobs):
        elapsed = time.perf_counter() - t0
        print(f'{i}/{len(jobs)}  norm={n_norm} copy={n_copy} skip={n_skip}  {elapsed/60:.1f} min')

yaml_body = {
    'path': str(dst_root).replace('\\', '/'),
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 2,
    'names': {0: 'Capilar', 1: 'microcotiledone'},
}
data_yaml.write_text(yaml.safe_dump(yaml_body, sort_keys=False), encoding='utf-8')
(dst_root / 'data.yaml').write_text(yaml.safe_dump(yaml_body, sort_keys=False), encoding='utf-8')
print('yaml:', data_yaml)
print(f'feito  norm={n_norm}  copiados_sem_tecido={n_copy}  ja_existiam={n_skip}')

referencia: ROSILHA-M-B_007_jpg.rf.405267d76c41ef9eec836746fdc1e97c_r1c1.jpg tissue_px 530181
HE [[     -0.006      -0.025]
 [      0.986       0.904]
 [      0.168       0.426]]
maxC [      1.362        1.69]
preview: ROSILHA-M-B_001_jpg.rf.e2db72f59c4cfe665_before_after.jpg normalized
preview: TOSTADA-B_001_jpg.rf.eea4a49cd9c74131acf_before_after.jpg normalized
preview: ZAZA-B_001_jpg.rf.4c782ac4019b662eda9a7a_before_after.jpg normalized
600/1620  norm=58 copy=0 skip=542  0.4 min
700/1620  norm=158 copy=0 skip=542  1.1 min
800/1620  norm=258 copy=0 skip=542  1.8 min
900/1620  norm=358 copy=0 skip=542  2.5 min
1000/1620  norm=458 copy=0 skip=542  3.2 min
1100/1620  norm=558 copy=0 skip=542  3.7 min
1200/1620  norm=658 copy=0 skip=542  4.3 min
1300/1620  norm=758 copy=0 skip=542  4.8 min
1400/1620  norm=858 copy=0 skip=542  5.4 min
1500/1620  norm=958 copy=0 skip=542  6.0 min
1600/1620  norm=1058 copy=0 skip=542  6.6 min
1620/1620  norm=1078 copy=0 skip=542  6.7 min
yaml: D:\projeto_pl

In [4]:
best_pt = (REPO_ROOT / CFG['project'] / CFG['run_name'] / 'weights' / 'best.pt').resolve()
n_train = len(list((CFG['dst_dataset'] / 'train' / 'images').glob('*.jpg')))
n_val = len(list((CFG['dst_dataset'] / 'valid' / 'images').glob('*.jpg')))
print('tiles', n_train, n_val)
assert n_train > 0 and n_val > 0, 'dataset Macenko vazio — rode a celula anterior'
assert pretrained.is_file(), pretrained

if best_pt.is_file():
    print('checkpoint ja existe, treino pulado:', best_pt)
else:
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model = YOLO(str(pretrained))
    train_results = model.train(
        data=str(data_yaml),
        epochs=CFG['epochs'],
        imgsz=CFG['imgsz'],
        batch=CFG['batch'],
        workers=CFG['workers'],
        patience=CFG['patience'],
        device=0 if torch.cuda.is_available() else 'cpu',
        project=str((REPO_ROOT / CFG['project']).resolve()),
        name=CFG['run_name'],
        exist_ok=True,
        classes=CFG['classes'],
        max_det=CFG['max_det'],
        amp=True,
        plots=False,
        retina_masks=False,
        overlap_mask=True,
        mask_ratio=4,
        lr0=0.01,
        lrf=0.01,
        degrees=90.0,
        flipud=0.5,
        fliplr=0.5,
        mosaic=0.15,
        copy_paste=0.15,
        mixup=0.0,
        close_mosaic=10,
        scale=0.3,
        hsv_s=0.7,
    )
    best_pt = Path(train_results.save_dir) / 'weights' / 'best.pt'

print('best.pt:', best_pt)
assert best_pt.is_file(), best_pt

tiles 1377 243
New https://pypi.org/project/ultralytics/8.4.160 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.155  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650 SUPER, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=[0], close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\data_capilar_tiled_macenko.yaml, degrees=90.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1

d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


train: Scanning D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3_macenko\train\labels... 1377 images, 13 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1377/1377 791.2it/s 1.7s0.1s
train: New cache created: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3_macenko\train\labels.cache


d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.10.0 ms, read: 941.6227.4 MB/s, size: 163.2 KB)
val: Scanning D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3_macenko\valid\labels... 243 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 243/243 939.8it/s 0.3s0.1s
val: New cache created: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3_macenko\valid\labels.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001667, momentum=0.9) with parameter groups 90 weight(decay=0.0), 101 weight(decay=0.0005), 100 bias(decay=0.0)
Using 1377 train, 243 val images for fraction=1.0 at imgsz=640
Using 0 dataloader workers
Logging results to D:\projeto_placentas_cl

In [5]:
import gc

CAPILAR_CLS = 0
val_images = CFG['dst_dataset'] / 'valid' / 'images'
val_labels = CFG['dst_dataset'] / 'valid' / 'labels'


def parse_gt_masks(label_path: Path, img_w: int, img_h: int, cls_keep: int = CAPILAR_CLS):
    masks = []
    if not label_path.exists():
        return masks
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        cls = int(float(parts[0]))
        if cls != cls_keep:
            continue
        coords = np.array([float(x.replace(',', '.')) for x in parts[1:]], dtype=np.float32).reshape(-1, 2)
        coords[:, 0] *= img_w
        coords[:, 1] *= img_h
        m = np.zeros((img_h, img_w), dtype=np.uint8)
        cv2.fillPoly(m, [coords.astype(np.int32)], 1)
        masks.append(m)
    return masks


def masks_from_result(r):
    if r.masks is None:
        return []
    return [(m > 0.5).astype(np.uint8) for m in r.masks.data.cpu().numpy()]


def iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union else 0.0


def greedy_match(pred_masks, gt_masks, thr: float):
    pairs = []
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            s = iou(pm, gm)
            if s >= thr:
                pairs.append((s, i, j))
    pairs.sort(reverse=True)
    used_p, used_g, matched = set(), set(), []
    for s, i, j in pairs:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        matched.append((s, i, j))
    return matched


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = YOLO(str(best_pt))
val_metrics = model.val(
    data=str(data_yaml),
    imgsz=CFG['imgsz'],
    batch=CFG['batch'],
    workers=CFG['workers'],
    device=0 if torch.cuda.is_available() else 'cpu',
    classes=CFG['classes'],
    plots=False,
    split='val',
)
map50 = float(val_metrics.seg.map50)
print(f'mask mAP50={map50:.4f}  v4={CFG["v4_map50"]:.3f}  delta={map50 - CFG["v4_map50"]:+.4f}')

img_paths = sorted(val_images.glob('*.jpg'))
tp = fp = fn = 0
ious = []
gt_area = pred_area = 0
for k, img_path in enumerate(img_paths, start=1):
    im = cv2.imread(str(img_path))
    h, w = im.shape[:2]
    gt_masks = parse_gt_masks(val_labels / f'{img_path.stem}.txt', w, h)
    r = model.predict(
        source=str(img_path),
        conf=CFG['conf'],
        imgsz=CFG['imgsz'],
        retina_masks=True,
        max_det=CFG['max_det'],
        classes=CFG['classes'],
        verbose=False,
    )[0]
    pred_masks = []
    for m in masks_from_result(r):
        if m.shape[0] != h or m.shape[1] != w:
            m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
        pred_masks.append(m)
    matches = greedy_match(pred_masks, gt_masks, CFG['iou_match'])
    tp += len(matches)
    fp += len(pred_masks) - len(matches)
    fn += len(gt_masks) - len(matches)
    ious.extend([s for s, _, _ in matches])
    gt_area += int(sum(int(m.sum()) for m in gt_masks))
    pred_area += int(sum(int(m.sum()) for m in pred_masks))
    if k % 50 == 0 or k == len(img_paths):
        print(f'F1 tiles {k}/{len(img_paths)}')

prec = tp / (tp + fp) if (tp + fp) else 0.0
rec = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
mean_iou = float(np.mean(ious)) if ious else 0.0
area_rel_err = abs(pred_area - gt_area) / gt_area if gt_area else 0.0
delta_f1 = f1 - CFG['v4_f1']

report = {
    'model': CFG['model_name'],
    'run_name': CFG['run_name'],
    'checkpoint': str(best_pt),
    'conf': CFG['conf'],
    'f1': f1,
    'mean_iou': mean_iou,
    'precision': prec,
    'recall': rec,
    'area_rel_error': area_rel_err,
    'mask_map50': map50,
    'v4_f1_at_0.33': CFG['v4_f1'],
    'v4_mean_iou': CFG['v4_iou'],
    'v4_mask_map50': CFG['v4_map50'],
    'delta_f1': delta_f1,
    'keep': bool(delta_f1 >= 0.02),
}
out_path = out_bench / 'macenko_vs_v4.json'
out_path.write_text(json.dumps(report, indent=2), encoding='utf-8')

print(f'F1@0.33={f1:.4f}  v4={CFG["v4_f1"]:.4f}  delta={delta_f1:+.4f}')
print(f'IoU={mean_iou:.4f}  v4={CFG["v4_iou"]:.4f}')
print(f'area_rel_err={area_rel_err:.4f}')
print('manter Macenko' if report['keep'] else 'manter v4')
print('report:', out_path)

Ultralytics 8.4.155  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650 SUPER, 4096MiB)
YOLO11s-seg summary (fused): 113 layers, 10,067,590 parameters, 0 gradients, 32.9 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1070.2217.3 MB/s, size: 165.9 KB)
val: Scanning D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3_macenko\valid\labels.cache... 243 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 243/243  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 243/243 25.5it/s 9.5s0.1s
                   all        243       5568      0.672      0.699      0.726      0.416      0.688      0.703      0.734      0.391
               Capilar        242       5568      0.672      0.699      0.726      0.416      0.688      0.703      0.734      0.391
Speed: 0.6ms preprocess, 17.4ms inference, 0.0ms loss, 2.8ms postprocess per image
mask mAP50=0.7342  v4=0.778